[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.02.notebook-title|Debrief -->
# Week 2 Debrief: Python fundamentals
<!-- /gen -->

*Worked answers for [Quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) 02: The Coffee Cart Ledger. Every tier, the [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) fix, and the week's [AI failure gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-failure-gallery).*

**Read this after attempting the [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest).** These are reference answers. Different loop names or print layouts are fine, if the numbers are right and you can defend your choices.

Run the data-setup cell first. It regenerates the quest's data with the same seed, `5813`, so every figure below is the figure your own quest should have produced.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
# Beans of Egham: One week of till records, as a list of dictionaries.
# Each record is ONE TRANSACTION (one customer order). Keys:
#   item        : The drink ordered
#   size        : "regular" or "large" (large costs £0.40 more per cup)
#   cups        : How many cups in the order (regulars buy rounds for the office)
#   price       : The price PER CUP in pounds, size included
#   time_of_day : When the sale happened, 24-hour clock, e.g. "08:35"
import os
import numpy as np

os.makedirs("assets/data/week02", exist_ok=True)  # where Bronze writes the founder's summary
rng = np.random.default_rng(5813)  # module seed: Do not change

menu = {
    "flat white": 3.40, "latte": 3.60, "cappuccino": 3.50,
    "espresso": 2.20, "mocha": 3.80, "hot chocolate": 3.20,
    "english breakfast tea": 2.40, "filter coffee": 2.80,
}
items = list(menu)
item_weights = [0.24, 0.17, 0.14, 0.08, 0.10, 0.07, 0.12, 0.08]
slot_hours = {"morning": [7, 8, 9, 10], "lunchtime": [11, 12, 13],
              "afternoon": [14, 15], "twilight": [16, 17, 18]}
slot_weights = [0.42, 0.30, 0.20, 0.08]
slot_names = list(slot_hours)

sales = []
for _ in range(150):
    item = items[int(rng.choice(len(items), p=item_weights))]
    size = "large" if rng.random() < 0.3 else "regular"
    if item == "latte":
        cups = int(rng.choice([1, 2, 3], p=[0.30, 0.45, 0.25]))
    else:
        cups = int(rng.choice([1, 2, 3], p=[0.72, 0.20, 0.08]))
    slot = slot_names[int(rng.choice(len(slot_names), p=slot_weights))]
    hour = int(rng.choice(slot_hours[slot]))
    minute = int(rng.integers(0, 60))
    price = menu[item] + (0.40 if size == "large" else 0.00)
    sales.append({
        "item": item,
        "size": size,
        "cups": cups,
        "price": round(price, 2),
        "time_of_day": f"{hour:02d}:{minute:02d}",
    })

print(f"Created sales: {len(sales)} till records from one week at Beans of Egham")
print(f"Example record: {sales[0]}")

## First look: What the four cells were telling you

- **[First look 1](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-first-look):** 150 records. Each record is a dictionary with the keys `item`, `size`, `cups`, `price`, and `time_of_day`
- **First look 2:** `price` is per cup. The second record is a round of 3 large espressos, so its line value is `cups * price` (3 × £2.60), not `price`
- **First look 3:** Counting transactions per item puts **flat white** on top
- **First look 4:** 55 of 150 transactions are rounds of 2+ cups, over a third. Transactions and cups are different quantities in this business. Both the Red Flags bug and the [Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze) distractor depend on that difference

## 🥉 Bronze: Worked answer

One loop builds a running revenue total and a dictionary of cups per item, using the counting pattern from the [Demonstration notebook](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02/Demonstration.ipynb). A second loop finds the item with the most cups.

In [ ]:
# 🥉 Bronze: Total revenue and best-selling drink, counted in cups.
total_revenue = 0
cups_per_item = {}

for sale in sales:
    total_revenue = total_revenue + sale["cups"] * sale["price"]
    item = sale["item"]
    cups_per_item[item] = cups_per_item.get(item, 0) + sale["cups"]

best_item = None
best_cups = 0
for item, cups in cups_per_item.items():
    if cups > best_cups:
        best_item = item
        best_cups = cups

print(f"Total weekly revenue: £{total_revenue:.2f}")
print(f"Best-selling drink:   {best_item} ({best_cups} cups)")

The cell below writes both numbers to `assets/data/week02/founder_summary.txt`, then reads the file back to prove it saved.

In [ ]:
# 🥉 Bronze: The founder's file. Write the summary, read it back to prove it saved.
with open("assets/data/week02/founder_summary.txt", "w") as file:
    file.write("Beans of Egham - week 1 till summary\n")
    file.write(f"Total weekly revenue: £{total_revenue:.2f}\n")
    file.write(f"Best-selling drink:   {best_item} ({best_cups} cups)\n")

with open("assets/data/week02/founder_summary.txt", "r") as file:
    print(file.read())

### The planted distractor

First look 3 showed **flat white** leading on transactions. Count cups instead, and latte leads, because the office rounds are mostly lattes. Cups are what "best-selling" means to someone ordering stock. An <!-- plain-language: everyday Analyst -->analyst who copies First look 3's loop without changing it to count the `cups` field ships the wrong best seller.

The cell below prints both counts side by side.

In [ ]:
# The distractor, made visible: Transactions vs cups, side by side.
transactions_per_item = {}
for sale in sales:
    item = sale["item"]
    transactions_per_item[item] = transactions_per_item.get(item, 0) + 1

for item in cups_per_item:
    transactions = transactions_per_item[item]
    cups = cups_per_item[item]
    print(f"{item}: {transactions} transactions, {cups} cups")

### The verification steps

- **Hand-check:** First look 2's first two records are 1 regular flat white at £3.40 and a round of 3 large espressos at £2.60 each. On a calculator: £3.40 + (3 × £2.60) = **£11.20**
- **Plausibility check:** Revenue per cup must sit between £2.20 and £4.20, the cheapest and most expensive prices on the menu
- **Cross-total check:** The values of `cups_per_item` must sum to the same cup total that the plausibility check uses. If they sum to 150, the dictionary counted transactions, not cups

The cell below runs all three.

In [ ]:
# Hand-check on sales[:2], then the plausibility check.
check_total = 0
for sale in sales[:2]:
    check_total = check_total + sale["cups"] * sale["price"]
print(f"Hand-check total for sales[:2]: £{check_total:.2f}  (calculator: £11.20)")

total_cups = 0
for sale in sales:
    total_cups = total_cups + sale["cups"]
print(f"Revenue per cup: £{total_revenue / total_cups:.2f}"
      f"  (must sit between £2.20 and £4.20)")

cups_via_dict = sum(cups_per_item.values())
print(f"Cross-total: cups_per_item sums to {cups_via_dict}"
      f"  (matches {total_cups} cups, not 150 transactions)")

## 🥈 Silver: Worked answer

Both drafts print the same slot totals, so the question is not which one is correct. Either choice earns [Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver) if you defend it:

- **Draft A (if/elif, running totals):** Draft A is shorter, holds only three numbers at a time, and uses only Week-1 Python
- **Draft B (a dictionary of lists):** Draft B keeps every transaction's line value. This means that transaction counts, averages, lowest values, and highest values need no extra work. Its `slot_for()` function keeps the founder's slot definitions in one place

Neither draft covers the founder's new twilight slot. B needs two small edits for twilight, one new key and one new branch in `slot_for()`. A needs the same change made inside its main loop. Choosing A and defending it on simplicity is a good Silver answer, and so is choosing B and defending it on adaptability. "I picked the one that ran" is the only failing answer, because both ran.

The cell below is Draft B, adapted.

In [ ]:
# 🥈 Silver: Draft B adapted for the founder's four slots.
def slot_for(time_of_day):
    hour = int(time_of_day.split(":")[0])
    if hour < 11:
        return "morning"
    elif hour < 14:
        return "lunchtime"
    elif hour < 16:
        return "afternoon"
    return "twilight"

lines_by_slot = {"morning": [], "lunchtime": [], "afternoon": [], "twilight": []}

for sale in sales:
    lines_by_slot[slot_for(sale["time_of_day"])].append(sale["cups"] * sale["price"])

for slot, lines in lines_by_slot.items():
    print(f"{slot}: £{sum(lines):.2f} ({len(lines)} transactions)")

The same change for Draft A is one new key in the dictionary and one new `elif hour < 16:` branch before the `else`.

The twilight row shows a dozen transactions and modest revenue. The revenue is small, but it is not £0.00. A draft that still sends every sale from 14:00 onwards to "afternoon" prints £0.00 for twilight, and that is a wrong answer. For the founder, two trial evenings of late opening earn little so far. The honest recommendation is "keep trialling and re-measure", not "twilight is a failure".

## 🥇 Gold: One worked example

Any clearly defined rule is a valid [Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold) answer, as long as you build it, run it, and compare its revenue with the as-is revenue. This worked example uses the first suggestion from the quest: **Rounds of 2+ cups get 10% off**.

In [ ]:
# 🥇 Gold rule: Rounds of 2+ cups get 10% off the whole transaction.
revenue_as_is = 0.0
revenue_with_rule = 0.0

for sale in sales:
    line = sale["cups"] * sale["price"]
    revenue_as_is = revenue_as_is + line
    if sale["cups"] >= 2:
        line = line * 0.90          # the 10% loyalty discount
    revenue_with_rule = revenue_with_rule + line

cost = revenue_as_is - revenue_with_rule
print(f"Revenue as-is:      £{revenue_as_is:.2f}")
print(f"Revenue under rule: £{revenue_with_rule:.2f}")
print(f"Cost of the scheme: £{cost:.2f}"
      f"  ({cost / revenue_as_is * 100:.1f}% of weekly revenue)")

The simulation answers the founder's question (*what would it cost me?*) with a number. The cost is a few per cent of weekly revenue, **assuming behaviour does not change**. State that assumption in your one-sentence rule. The scheme exists to make customers buy more rounds, more often. One week of data cannot tell you whether it would work. A Gold answer prints two totals *and* names that limitation.

## 🚩 The bug and the fix ("Red Flags")

**The bug:** `cups_sold = len(sales)` counts transactions and labels them cups. First look 4 showed 55 transactions were rounds of 2+ cups, so the true cup count is far higher. "Average per cup" (revenue ÷ transactions) comes out around £4.85, more than the most expensive drink on the menu. The plausibility check catches it.

**In business terms:** If this summary ships, the founder orders stock for roughly a third fewer drinks than she sells. She runs out of cups, lids, and milk midweek. She also walks into her supplier meeting believing customers pay about £4.85 a cup. The most expensive drink on the menu costs £4.20, so the supplier will not take that figure seriously.

**A directing prompt that fixes it:** *"This summary counts transactions with `len(sales)` and calls them cups, but one transaction can be several cups. Fix it so cups are counted by summing the `cups` field, keep everything else the same, and explain the difference between the two counts."*

The cell below is the fixed summary.

In [ ]:
# 🚩 Fixed summary: Cups counted by summing the cups field.
total_revenue = 0
cups_sold = 0
for sale in sales:
    total_revenue = total_revenue + sale["cups"] * sale["price"]
    cups_sold = cups_sold + sale["cups"]

average_per_cup = total_revenue / cups_sold

print("=== Beans of Egham: the week in numbers ===")
print(f"Transactions:     {len(sales)}")
print(f"Cups sold:        {cups_sold}")
print(f"Total revenue:    £{total_revenue:.2f}")
print(f"Average per cup:  £{average_per_cup:.2f}")

The fixed version reports both counts with a clear label on each. Transactions and cups are both useful, as long as neither is given the other's name.

## 🤖 AI failure gallery

Three realistic ways an [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) gets this week's material wrong. Each entry gives the plausible AI output, why it is wrong in business terms, and the catch (what would have warned you).

### Failure 1: The hallucinated string method

Asked to "tidy the drink names for the founder's report", the Associate writes the cell below with a British spelling. Run it. It fails.

In [ ]:
# Associate draft: 'capitalise each drink name for the report'
item = "flat white"
print(item.capitalise())   # plausible: It even spells it the British way

**Why it is wrong:** There is no `.capitalise()`. The real method is `.capitalize()`, with the American spelling, and `.title()` capitalises every word.

**In business terms:** This one fails loudly, with an `AttributeError` five minutes before the founder's meeting. The more dangerous version is a hallucinated argument that runs and silently does nothing (the [AI Guide, section 2](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/ai-guide.md) has one). Either way, the report is late or wrong.

**The catch:** Run the line. Never ship a cell you have not run. When an Associate names a method you have not met, check it exists: `print(dir(""))` lists every method strings really have. The [string methods documentation](https://docs.python.org/3/library/stdtypes.html#string-methods) is the final authority when you and your Associate disagree.

### Failure 2: The off-by-one `range()` loop

Asked for a revenue total "using indexes so I can see the position of each sale", the Associate writes the loop below. It runs without error and gives the wrong answer.

In [ ]:
# Associate draft: Total revenue 'by index'. It runs. It is wrong.
total = 0
for i in range(1, len(sales)):     # starts at 1: LOOKS harmless
    total = total + sales[i]["cups"] * sales[i]["price"]

print(f"Total revenue: £{total:.2f}")   # quest answer minus the FIRST transaction

**Why it is wrong:** Python counts positions from 0, so `range(1, len(sales))` skips `sales[0]`. The correct pattern is `range(len(sales))`, or better, `for sale in sales:` with no indexes at all.

**In business terms:** The week's revenue is understated by exactly one transaction, £3.40 here. That error is small enough to go unnoticed, but large enough to make the founder's till roll and your report disagree. If the total is wrong by £3.40, which other numbers are wrong?

**The catch:** Cross-check against a no-index loop. The two must agree to the penny. Or count what the loop handles. It reaches 149 records while `len(sales)` says 150. That is the whole bug in one line. [PREDICT-RUN](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-predict-run) on `range(1, 6)` in the [Demonstration notebook](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02/Demonstration.ipynb) is the same lesson on a smaller scale.

### Failure 3: The f-string that silently rounds

Asked to "print a clean headline figure for the slide", the Associate writes the cell below. The printed figure is not the true total.

In [ ]:
# Associate draft: 'clean' headline numbers for the founder's slide.
total_revenue = 0
for sale in sales:
    total_revenue = total_revenue + sale["cups"] * sale["price"]

print(f"Headline for the slide:  £{total_revenue:.0f}")     # rounds: Silently
print(f"What the till roll says: £{total_revenue:.2f}")     # the actual total

**Why it is wrong:** `:.0f` does not cut the pennies off. It rounds the display. The number underneath is untouched, but the reader does not see the true total. No error appears, and the two printed lines disagree with each other.

**In business terms:** The founder's slide says one figure and her till reconciliation says another, 10p apart. To Finance or a supplier, an unexplained gap between two "totals" reads as money missing. Every number in the deck then gets a second check. Round if you like, but say so on the slide ("to the nearest pound"), and reconcile to the penny.

**The catch:** Formatting changes presentation, never value. Print any figure that must reconcile with `:.2f`, and label any rounded figure as rounded. If two of your own printouts of "the total" differ, believe neither until you know why.

## Closing the ledger

Three habits from this week:

1. **Count the right thing.** Transactions, cups, items, and customers are four different counts, and choosing what to count is a business <!-- plain-language: everyday decision -->decision before it is a technical one
2. **Verify with checks the client would respect.** A hand-check on two records and a plausibility bound ("no cup costs more than £4.20") caught every bug in this quest
3. **Judge drafts on their future, not only their output.** Both Silver drafts printed the same numbers, but they differed in what they would cost you next week

Update your `QuestLog.md` and your [AI Use Record](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-use-record).

**Next week:** The assessment briefing covers both assessments and the [interpretation document](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/interpretation.md), which explains what markers read.